In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [3]:
import pandas as pd
import numpy as np
import torch
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    PreTrainedTokenizerBase
)
from transformers.utils import PaddingStrategy
from datasets import Dataset
from typing import Optional, Union

# Parameter-Efficient Fine-Tuning (PEFT) Imports
from peft import LoraConfig, get_peft_model, TaskType

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

def compute_metrics(eval_predictions):
    predictions, label_ids = eval_predictions
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": (preds == label_ids).astype(np.float32).mean().item()}

def run_pipeline(train_df: pd.DataFrame, test_df: pd.DataFrame, model_name: str = "microsoft/deberta-v3-base") -> pd.DataFrame:
    options = ['A', 'B', 'C', 'D', 'E']
    option_to_index = {opt: i for i, opt in enumerate(options)}
    index_to_option = {i: opt for i, opt in enumerate(options)}

    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df['label'] = train_df['answer'].map(option_to_index)
    test_df['label'] = 0 

    train_df, eval_df = train_test_split(
        train_df, 
        test_size=0.1, 
        stratify=train_df['label'], 
        random_state=42
    )

    train_ds = Dataset.from_pandas(train_df)
    eval_ds = Dataset.from_pandas(eval_df)
    test_ds = Dataset.from_pandas(test_df)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def preprocess_function(examples):
        """
        Data Formatting Strategy:
        We explicitly concatenate the Question (prompt) with each Option pair-wise.
        Format: [CLS] Prompt [SEP] Option [SEP]
        """
        first_sentences = [[str(context)] * 5 for context in examples["prompt"]]
        second_sentences = [[str(examples[opt][i]) for opt in options] for i in range(len(examples["prompt"]))]
        
        first_sentences = sum(first_sentences, [])
        second_sentences = sum(second_sentences, [])
        
        tokenized_examples = tokenizer(
            first_sentences,
            second_sentences,
            truncation="only_first",  
            max_length=512,           
            padding=False
        )
        return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

    train_cols_to_remove = [c for c in train_ds.column_names if c != 'label']
    tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_cols_to_remove)
    
    eval_cols_to_remove = [c for c in eval_ds.column_names if c != 'label']
    tokenized_eval = eval_ds.map(preprocess_function, batched=True, remove_columns=eval_cols_to_remove)

    test_cols_to_remove = [c for c in test_ds.column_names if c != 'label']
    tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=test_cols_to_remove)

    # 1. Initialize Base Model
    model = AutoModelForMultipleChoice.from_pretrained(model_name)

    # 2. Configure and Apply LoRA
    # By targeting the attention projections (query, value), we inject low-rank matrices 
    # that adapt the model's contextual understanding without modifying the base weights.
    lora_config = LoraConfig(
        r=8,                                   # Rank of the update matrices
        lora_alpha=16,                         # Scaling factor for LoRA updates
        target_modules=["query_proj", "value_proj"], # Target DeBERTa attention layers
        lora_dropout=0.1,                      # Dropout probability for LoRA layers
        bias="none",
        task_type=TaskType.SEQ_CLS             # PEFT handles MultipleChoice under Sequence Classification logic
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()         # Logs the % of parameters being trained (typically < 1%)

    bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    # 3. Memory & Efficiency Strategies via TrainingArguments
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",             
        save_strategy="epoch",             
        logging_strategy="epoch",          
        load_best_model_at_end=True,       
        metric_for_best_model="eval_loss",
        learning_rate=1e-4,                # LoRA requires a higher learning rate (1e-4 to 5e-4) than full fine-tuning
        per_device_train_batch_size=4,     # Physical batch size (constrained by VRAM)
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,     # Effective Batch Size = 16 (4 physical * 4 accum)
        gradient_checkpointing=True,       # CRITICAL for memory: drops intermediate activations
        warmup_ratio=0.1,                  
        lr_scheduler_type="cosine",        
        num_train_epochs=25,                
        weight_decay=0.01,
        fp16=False,
        bf16=bf16_supported,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Inference logic
    predictions = trainer.predict(tokenized_test).predictions
    
    top_3_indices = np.argsort(predictions, axis=1)[:, ::-1][:, :3]
    
    top_3_predictions = []
    for indices in top_3_indices:
        pred_str = " ".join([index_to_option[idx] for idx in indices])
        top_3_predictions.append(pred_str)

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Prediction': top_3_predictions
    })

    return submission_df

if __name__ == "__main__":
    pass

In [4]:
submission = run_pipeline(train_df, test_df, model_name="microsoft/deberta-v3-base")
submission.to_csv("submission.csv", index=False)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                  

trainable params: 295,681 || all params: 184,718,594 || trainable%: 0.1601


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,12.650665,3.177734,0.270000
2,12.621676,3.150391,0.290000
3,12.540604,3.080078,0.400000
4,12.081860,2.853516,0.440000
5,11.326754,2.562500,0.495000
6,10.578793,2.388672,0.490000
7,10.016859,2.251953,0.550000
8,9.377690,2.191406,0.630000
9,9.054396,1.924805,0.645000
10,8.700427,1.835938,0.665000


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector